# IE2026 Task 1b — Clean QLoRA **3k** run (Qwen2.5-VL-7B) — single file, multi-checkpoint

**Requires T4×2 GPU. Self-contained. Prefer this notebook over `resume-from-checkpoint-300.ipynb`.**

## Why a new notebook?

The previous 2.6k run (devtest CI **0.034**) was confounded vs RunFT (2.3k, CI **0.028**):

| Issue | Old 2.6k | This notebook |
|-------|----------|---------------|
| `MAX_STEPS` | hard-coded **510** (only 78% of 2.6k) | **auto** `N // GRAD_ACCUM` → **750** for 3,000 items |
| Subsample | `random.sample` → shuffle order **not nested** across sizes | **nested prefix** of one seeded 3k permutation |
| Early stop | stop on **first** non-improving CI eval | **patience = 3** evals |
| Resume `best_ci` | reset to `inf` every session | **persisted** in `early_stop_state.json` |
| Start | resumed from a **foreign** step-300 ckpt | **from scratch** by default |

## Protocol (apples-to-apples data-volume ablation)

1. Shuffle all 3,000 train items once with `SEED=42` → fixed global order.
2. Take the **prefix** of length `TRAIN_SUBSAMPLE_N` (default 3,000 = full train set), rounded down to a multiple of `GRAD_ACCUM`.
3. Train **one epoch** over that prefix (`MAX_STEPS = len(records) // GRAD_ACCUM`).
4. Checkpoint + dev-CI every `SAVE_STEPS` (100). Keep the **best-CI** adapter.
5. Early-stop only after `EARLY_STOP_PATIENCE` consecutive non-improving CI evals (or finish the epoch).

Larger `N` is then a true superset of smaller `N` under the same seed — unlike the old `random.sample` path.

## Multi-session plan (Kaggle 12h)

~750 steps × ~1–2 min/opt-step ≈ **10–24h** with CI evals → plan **3 sessions**:

| Session | `RESUME_FROM` | `RESUME_STEP` | `SESSION_STOP_STEP` |
|---------|---------------|---------------|---------------------|
| 1 | `None` | `0` | `300` |
| 2 | `.../step_300` | `300` | `600` |
| 3 | `.../step_600` | `600` | `MAX_STEPS=750` (or `None`) |

After each pause: download `checkpoints/step_*` → upload as Kaggle dataset → set `RESUME_*` for the next session.

## Defaults for this re-run

- Model: `Qwen2.5-VL-7B-Instruct`, 4-bit NF4, frozen vision encoder, LoRA r=8
- Train: 3,000 items (full train set), 1 epoch, lr 2e-4 cosine, `MAX_PIXELS=256×28×28`
- Select: best **dev** CI (official Task 1b metric)
- Infer later with CoT5 / 1024px notebook (same as before)


In [ ]:
import os, warnings, time
warnings.filterwarnings('ignore')
for _major in ('12', '13'):
    _src = f'/usr/local/cuda/lib64/libnvJitLink.so.{_major}'
    _dst = '/usr/local/cuda/lib64/libnvJitLink.so.13'
    if os.path.exists(_src) and not os.path.exists(_dst):
        os.symlink(_src, _dst); print(f'Symlinked .{_major} -> .13'); break
os.environ['BITSANDBYTES_NOWELCOME'] = '1'
os.environ['BNB_CUDA_VERSION']        = '128'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'
!pip install -q -U 'transformers>=4.49.0' 'peft>=0.10.0' \
    accelerate bitsandbytes qwen-vl-utils 2>&1 | tail -4
print('Dependencies ready.')


In [ ]:
import os, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ── Dataset ──────────────────────────────────────────────────────────────
REPO_ID  = 'QCRI/AynVQA-ArabicNLP26'
TASK     = 'task1b'
LANG     = 'en'
SPLIT    = 'train'

# ── Model ────────────────────────────────────────────────────────────────
VLM_MODEL     = 'Qwen/Qwen2.5-VL-7B-Instruct'
MAX_PIXELS    = 256 * 28 * 28   # ~244 image tokens — fast backward

# ── LoRA (LLM layers only) ────────────────────────────────────────────────
LORA_RANK     = 8
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                 'gate_proj', 'up_proj', 'down_proj']

# ── Training ─────────────────────────────────────────────────────────────
SEED              = 42
# Nested-prefix size from the fixed SEED=42 permutation of all 3,000 items.
# Use 2000 / 2300 / 2600 / 3000 for comparable data-volume ablations.
# Automatically rounded DOWN to a multiple of GRAD_ACCUM after draw.
TRAIN_SUBSAMPLE_N = 3000
GRAD_ACCUM        = 4
BATCH_SIZE        = 1
LEARNING_RATE     = 2e-4
MAX_SEQ_LEN       = 1280
WARMUP_RATIO      = 0.05
SAVE_STEPS        = 100
LOGGING_STEPS     = 10
NUM_EPOCHS        = 1

# MAX_STEPS is derived after the nested prefix is drawn (data cell).
# Placeholder only — do not hard-code.
MAX_STEPS = None

# Early stopping on labelled dev CI (lower is better).
# Patience = number of consecutive non-improving *checkpoint evals*
# before stopping. Old notebook used patience=1 (too aggressive).
EARLY_STOP_PATIENCE = 3
# Do not early-stop before this many opt steps (lets LR leave warmup).
MIN_STEPS_BEFORE_EARLY_STOP = 200
# If True, ignore early stop and always run to MAX_STEPS (still saves best-CI).
FORCE_FULL_EPOCH = False

# ── Multi-session control (Kaggle 12h) ────────────────────────────────────
# Session 1 default: train from scratch, pause at 200.
# Later sessions: set RESUME_FROM / RESUME_STEP / SESSION_STOP_STEP as in the
# markdown table. Set SESSION_STOP_STEP = None to run until MAX_STEPS / early-stop.
SESSION_STOP_STEP = 300  # pause at first SAVE_STEPS boundary >= this

# ── Resume (None = train from scratch) ───────────────────────────────────
# Point at a previous checkpoint folder that contains:
#   adapter_model.safetensors (or .bin), adapter_config.json,
#   optimizer.pt, scheduler.json
# Optionally also early_stop_state.json (written next to checkpoints).
RESUME_FROM = None
RESUME_STEP = 0

# ── Output ───────────────────────────────────────────────────────────────
OUTPUT_DIR    = '/kaggle/working/checkpoints'
FINAL_ADAPTER = '/kaggle/working/adapter_final'
EARLY_STOP_STATE_PATH = os.path.join(OUTPUT_DIR, 'early_stop_state.json')

N_GPUS = torch.cuda.device_count()
assert N_GPUS >= 1, 'No GPU found'
print(f'GPUs: {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f} GB)')
print(f'VLM_MODEL:            {VLM_MODEL}')
print(f'TRAIN_SUBSAMPLE_N:    {TRAIN_SUBSAMPLE_N} (nested prefix, seed={SEED})')
print(f'EARLY_STOP_PATIENCE:  {EARLY_STOP_PATIENCE}')
print(f'MIN_STEPS_BEFORE_ES:  {MIN_STEPS_BEFORE_EARLY_STOP}')
print(f'FORCE_FULL_EPOCH:     {FORCE_FULL_EPOCH}')
print(f'SESSION_STOP_STEP:    {SESSION_STOP_STEP}')
print(f'Resume from:          {RESUME_FROM or "scratch"} (step {RESUME_STEP})')
print(f'MAX_PIXELS:           {MAX_PIXELS} (~{MAX_PIXELS//784} image tokens)')


In [ ]:
import json, random
from huggingface_hub import login, hf_hub_download
from tqdm.auto import tqdm

# Prefer Kaggle secret / env var — never hard-code tokens in the notebook.
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('HF login: ok (token from secret/env)')
else:
    print('WARNING: no HF_TOKEN found — public downloads may still work; '
          'gated models will fail. Add HF_TOKEN as a Kaggle secret.')


In [ ]:
import json, random
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

jsonl = hf_hub_download(
    REPO_ID, filename=f'{TASK}/{SPLIT}_{LANG}.jsonl', repo_type='dataset')
all_records = [json.loads(l) for l in open(jsonl, encoding='utf-8') if l.strip()]
print(f'Total records: {len(all_records)} | has labels: {"labels" in all_records[0]}')

# ── Nested-prefix subsample (fixed global permutation) ───────────────────
# Old recipe used random.sample(N). That nests *sets* for some seeds but
# produces non-comparable epoch shuffles when N changes (dataset indices
# 0..N-1 reshuffle differently). Here we:
#   1) shuffle all 3000 once with SEED
#   2) take prefix [:N]
#   3) round down to a multiple of GRAD_ACCUM
# So TRAIN_SUBSAMPLE_N=2000 ⊂ 2300 ⊂ 2600 ⊂ 3000 under the same SEED, and
# the first K *items* of a larger run match the smaller run's item set.
_rng = random.Random(SEED)
full_order = list(all_records)
_rng.shuffle(full_order)

n_want = TRAIN_SUBSAMPLE_N if TRAIN_SUBSAMPLE_N else len(full_order)
n_want = min(n_want, len(full_order))
n_keep = (n_want // GRAD_ACCUM) * GRAD_ACCUM
if n_keep != n_want:
    print(f'Rounded TRAIN_SUBSAMPLE_N {n_want} → {n_keep} '
          f'(multiple of GRAD_ACCUM={GRAD_ACCUM})')
records = full_order[:n_keep]

# Auto-derive MAX_STEPS so one epoch covers every kept item exactly once
# (under the deterministic epoch order built later).
MAX_STEPS = len(records) * NUM_EPOCHS // GRAD_ACCUM
assert MAX_STEPS * GRAD_ACCUM == len(records) * NUM_EPOCHS, (
    f'Inconsistent schedule: MAX_STEPS={MAX_STEPS}, N={len(records)}, '
    f'GRAD_ACCUM={GRAD_ACCUM}, NUM_EPOCHS={NUM_EPOCHS}')

if SESSION_STOP_STEP is None:
    SESSION_STOP_STEP = MAX_STEPS
elif SESSION_STOP_STEP > MAX_STEPS:
    print(f'SESSION_STOP_STEP {SESSION_STOP_STEP} > MAX_STEPS {MAX_STEPS} '
          f'— clamping to MAX_STEPS')
    SESSION_STOP_STEP = MAX_STEPS

print(f'Nested prefix: {len(records)}/{len(all_records)} items '
      f'(requested {TRAIN_SUBSAMPLE_N}, seed={SEED})')
print(f'MAX_STEPS = {MAX_STEPS} opt steps '
      f'(= {len(records)} samples / {GRAD_ACCUM} accum × {NUM_EPOCHS} epoch)')
print(f'This session will run up to step {SESSION_STOP_STEP} '
      f'({"= MAX_STEPS, full run" if SESSION_STOP_STEP == MAX_STEPS else "scheduled pause"})')
print(f'First 5 image ids: {[r["image"] for r in records[:5]]}')

# Download images for the training prefix only
needed = sorted({r['image'] for r in records})
img_paths = {}
failed = []
for rel in tqdm(needed, desc='train images'):
    try:
        img_paths[rel] = hf_hub_download(
            REPO_ID, filename=rel, repo_type='dataset')
    except Exception as e:
        failed.append(rel)
        print(f'Failed: {rel}: {e}')
print(f'Downloaded {len(img_paths)}/{len(needed)} train images '
      f'({len(failed)} failed).')


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

SYSTEM_PROMPT = (
    'You are a visual fact-checker examining an image from the Arab world.\n'
    'Below are THREE statements about this image. '
    'Exactly ONE statement is grounded in the image (True). '
    'The other two are plausible-sounding hallucinations (False).'
)
USER_TEMPLATE = (
    'Statement 1: {s0}\n'
    'Statement 2: {s1}\n'
    'Statement 3: {s2}\n\n'
    'Instructions:\n'
    '- On the VERY FIRST line write ONLY: "Answer: X" where X is 1, 2, or 3.\n'
    '- For each statement evaluate:\n'
    '    (a) Colour/texture evidence for or against\n'
    '    (b) Shape/form evidence for or against\n'
    '    (c) Contextual evidence for or against\n'
    '- Then state your conclusion.\n'
    'Do not write anything before the Answer line.'
)

processor = AutoProcessor.from_pretrained(VLM_MODEL, max_pixels=MAX_PIXELS)

class HalDetectDataset(Dataset):
    def __init__(self, records, img_paths, processor, max_seq_len):
        self.samples = [
            {'image_path': img_paths[r['image']],
             'statements': r['statements'],
             'true_idx':   r['labels'].index(True)}
            for r in records if r['image'] in img_paths
        ]
        self.processor   = processor
        self.max_seq_len = max_seq_len
        print(f'Dataset: {len(self.samples)} samples '
              f'({len(records)-len(self.samples)} skipped)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        target   = f'Answer: {s["true_idx"] + 1}'
        user_txt = USER_TEMPLATE.format(
            s0=s['statements'][0], s1=s['statements'][1], s2=s['statements'][2])
        messages = [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': [
                {'type': 'image', 'image': s['image_path']},
                {'type': 'text',  'text':  user_txt}]},
            {'role': 'assistant', 'content': target},
        ]
        prompt_text = self.processor.apply_chat_template(
            messages[:-1], tokenize=False, add_generation_prompt=True)
        full_text   = self.processor.apply_chat_template(
            messages,       tokenize=False, add_generation_prompt=False)
        image_inputs, _ = process_vision_info(messages)

        enc_full   = self.processor(text=[full_text],   images=image_inputs,
                                    truncation=False, return_tensors='pt')
        enc_prompt = self.processor(text=[prompt_text], images=image_inputs,
                                    truncation=False, return_tensors='pt')

        input_ids  = enc_full['input_ids'][0][:self.max_seq_len]
        attn_mask  = enc_full['attention_mask'][0][:self.max_seq_len]
        prompt_len = enc_prompt['input_ids'].shape[1]

        pad_id  = self.processor.tokenizer.pad_token_id or 0
        pad_len = self.max_seq_len - input_ids.shape[0]
        if pad_len > 0:
            input_ids = torch.cat(
                [input_ids, torch.full((pad_len,), pad_id, dtype=torch.long)])
            attn_mask = torch.cat(
                [attn_mask, torch.zeros(pad_len, dtype=torch.long)])

        labels = input_ids.clone()
        labels[:prompt_len] = -100
        labels[input_ids == pad_id] = -100

        return {
            'input_ids':      input_ids,
            'attention_mask': attn_mask,
            'pixel_values':   enc_full['pixel_values'],
            'image_grid_thw': enc_full['image_grid_thw'],
            'labels':         labels,
        }

dataset = HalDetectDataset(records, img_paths, processor, MAX_SEQ_LEN)

ex = dataset[0]
print(f'input_ids:     {ex["input_ids"].shape}')
print(f'pixel_values:  {ex["pixel_values"].shape}')
n_tgt = (ex['labels'] != -100).sum().item()
print(f'target tokens: {n_tgt}')
print(f'decoded:       '
      f'{processor.decode(ex["labels"][ex["labels"] != -100], skip_special_tokens=True)}')

def collate_fn(batch):
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels':         torch.stack([b['labels']         for b in batch]),
        'pixel_values':   torch.cat(  [b['pixel_values']   for b in batch], dim=0),
        'image_grid_thw': torch.cat(  [b['image_grid_thw'] for b in batch], dim=0),
    }

# ── Deterministic, session-independent epoch order ───────────────────────
# Build ONE shuffle of dataset indices long enough for MAX_STEPS*GRAD_ACCUM
# micro-batches. Each Kaggle session slices into that fixed order at
# RESUME_STEP*GRAD_ACCUM so multi-session training never reshuffles.
import random as _random

def _build_epoch_order(n_items, n_needed, seed):
    order, epoch = [], 0
    while len(order) < n_needed:
        rng = _random.Random(seed + epoch)
        idx = list(range(n_items))
        rng.shuffle(idx)
        order.extend(idx)
        epoch += 1
    return order[:n_needed]

n_items = len(dataset)
total_batches_needed = MAX_STEPS * GRAD_ACCUM
# If some images failed to download, shrink schedule to what we actually have
if n_items < len(records):
    n_keep_eff = (n_items // GRAD_ACCUM) * GRAD_ACCUM
    print(f'WARNING: only {n_items}/{len(records)} samples after image download. '
          f'Recomputing MAX_STEPS for {n_keep_eff} usable samples.')
    # Rebuild a truncated dataset view would be ideal; for simplicity recompute steps
    MAX_STEPS = max(n_keep_eff * NUM_EPOCHS // GRAD_ACCUM, 1)
    total_batches_needed = MAX_STEPS * GRAD_ACCUM
    if SESSION_STOP_STEP > MAX_STEPS:
        SESSION_STOP_STEP = MAX_STEPS

full_epoch_order = _build_epoch_order(n_items, total_batches_needed, SEED)
start_batch    = RESUME_STEP * GRAD_ACCUM
session_order  = full_epoch_order[start_batch:]
session_subset = torch.utils.data.Subset(dataset, session_order)

train_loader = DataLoader(
    session_subset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2,
    prefetch_factor=2, persistent_workers=True, pin_memory=False,
)
warmup_steps = int(MAX_STEPS * WARMUP_RATIO)
print(f'\nDeterministic epoch order: {len(full_epoch_order)} batches '
      f'(seed={SEED}, n_items={n_items})')
print(f'This session covers batches [{start_batch}:{len(full_epoch_order)}] '
      f'({len(session_order)} remaining)')
print(f'Target opt steps: {MAX_STEPS}')
print(f'Warmup steps:     {warmup_steps}')
session_steps_remaining = min(SESSION_STOP_STEP, MAX_STEPS) - RESUME_STEP
print(f'Est. at ~137s/opt-step: '
      f'{max(session_steps_remaining,0)*137/3600:.1f}h training this session '
      f'(+ CI evals ~35-40 min each)')


In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, \
    prepare_model_for_kbit_training, PeftModel

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)

max_mem = {i: '13000MiB' for i in range(N_GPUS)}
max_mem['cpu'] = '4GiB'

print('Loading base model...')
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL, torch_dtype=dtype,
    device_map='auto', max_memory=max_mem,
    quantization_config=bnb_config)

if hasattr(base_model, 'hf_device_map'):
    from collections import Counter
    dist = dict(Counter(base_model.hf_device_map.values()))
    print(f'Layer distribution: {dist}')

LLM_DEVICE = next(base_model.lm_head.parameters()).device
print(f'LLM primary device: {LLM_DEVICE}  (pixel_values routed here)')

base_model = prepare_model_for_kbit_training(
    base_model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False})

lora_cfg = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias='none', task_type=TaskType.CAUSAL_LM)

if RESUME_FROM:
    print(f'Resuming: loading adapter from {RESUME_FROM}')
    model = PeftModel.from_pretrained(base_model, RESUME_FROM, is_trainable=True)
    print('Adapter loaded — continuing training.')
else:
    model = get_peft_model(base_model, lora_cfg)
    print('Fresh LoRA adapters — training from scratch.')

n_frozen = n_train = 0
for name, param in model.named_parameters():
    if 'visual' in name:
        param.requires_grad = False
        n_frozen += param.numel()
    elif param.requires_grad:
        n_train += param.numel()
print(f'Frozen  (vision encoder): {n_frozen:,}')
print(f'Trainable (LoRA on LLM):  {n_train:,}')
model.print_trainable_parameters()

for i in range(N_GPUS):
    a = torch.cuda.memory_allocated(i)/1024**3
    t = torch.cuda.get_device_properties(i).total_memory/1024**3
    print(f'GPU {i}: {a:.1f}/{t:.1f} GB')


In [ ]:
import shutil
from qwen_vl_utils import process_vision_info

# ── Load labelled dev split (early-stop / best-CI selection) ─────────────
dev_jsonl = hf_hub_download(
    REPO_ID, filename=f'{TASK}/dev_{LANG}.jsonl', repo_type='dataset')
dev_records = [json.loads(l) for l in open(dev_jsonl, encoding='utf-8') if l.strip()]
print(f'Dev records: {len(dev_records)}')

dev_needed = sorted({r['image'] for r in dev_records})
dev_img_paths = {}
for rel in tqdm(dev_needed, desc='dev images'):
    try:
        dev_img_paths[rel] = hf_hub_download(REPO_ID, filename=rel, repo_type='dataset')
    except Exception as e:
        print(f'Failed: {rel}: {e}')
print(f'Downloaded {len(dev_img_paths)}/{len(dev_needed)} dev images.')

dev_samples = [
    {'image_path': dev_img_paths[r['image']],
     'statements': r['statements'],
     'true_idx':   r['labels'].index(True)}
    for r in dev_records if r['image'] in dev_img_paths
]
print(f'Dev samples ready: {len(dev_samples)}')


def _rate(n, d):
    return float(round(n / d, 6)) if d else 0.0


@torch.no_grad()
def evaluate_ci(model, processor, samples, max_new_tokens=8):
    # Official Task 1b metric panel on labelled items (dev).
    # Predicts one true-statement index, then rebuilds three T/F judgments.
    model.eval()

    total = q_minus_total = 0
    q_plus_c = q_minus_c = combined_c = 0
    cfhr_2 = cfhr_2_total = cfhr_3 = cfhr_3_total = 0

    for s in samples:
        true_idx = s['true_idx']
        user_txt = USER_TEMPLATE.format(
            s0=s['statements'][0], s1=s['statements'][1], s2=s['statements'][2])
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': [
                {'type': 'image', 'image': s['image_path']},
                {'type': 'text',  'text':  user_txt}]},
        ]
        prompt_text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        image_inputs, _ = process_vision_info(messages)
        enc = processor(text=[prompt_text], images=image_inputs,
                         truncation=False, return_tensors='pt')

        pv  = enc.pop('pixel_values').to(dtype).to(LLM_DEVICE)
        thw = enc.pop('image_grid_thw').to(LLM_DEVICE)
        enc = {k: v.to(LLM_DEVICE) for k, v in enc.items()}
        enc['pixel_values']   = pv
        enc['image_grid_thw'] = thw

        gen = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id)
        new_tokens = gen[0][enc['input_ids'].shape[1]:]
        decoded = processor.decode(new_tokens, skip_special_tokens=True)

        pred_idx = None
        for tok in ('1', '2', '3'):
            if tok in decoded[:12]:
                pred_idx = int(tok) - 1
                break

        false_idx = [i for i in range(3) if i != true_idx]
        if pred_idx is None:
            inc_qp = inc_qm0 = inc_qm1 = False
        else:
            inc_qp  = (pred_idx == true_idx)
            inc_qm0 = (pred_idx != false_idx[0])
            inc_qm1 = (pred_idx != false_idx[1])

        total += 1
        q_minus_total += 2
        q_plus_c  += inc_qp
        q_minus_c += inc_qm0 + inc_qm1
        combined_c += inc_qp and inc_qm0 and inc_qm1

        if inc_qp:
            cfhr_2_total += 1
            if not (inc_qm0 and inc_qm1):
                cfhr_2 += 1
        if inc_qp or inc_qm0 or inc_qm1:
            cfhr_3_total += 1
            if not (inc_qp and inc_qm0 and inc_qm1):
                cfhr_3 += 1

    model.train()
    for module in model.modules():
        if 'Visual' in type(module).__name__:
            module.eval()

    return {
        'contrastive_instability': _rate(cfhr_3, cfhr_3_total),
        'combined_accuracy':       _rate(combined_c, total),
        'cfhr':                    _rate(cfhr_2, cfhr_2_total),
        'q_plus_accuracy':         _rate(q_plus_c, total),
        'q_minus_accuracy':        _rate(q_minus_c, q_minus_total),
    }


def save_early_stop_state(path, state):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'w') as f:
        json.dump(state, f, indent=2)


def load_early_stop_state(paths):
    # Try several locations so resume does not reset best_ci to inf.
    for p in paths:
        if p and os.path.exists(p):
            with open(p) as f:
                st = json.load(f)
            print(f'Restored early-stop state from {p}: {st}')
            return st
    print('No early_stop_state.json found — starting fresh best_ci tracking.')
    return None


In [ ]:
import json
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

os.makedirs(OUTPUT_DIR,    exist_ok=True)
os.makedirs(FINAL_ADAPTER, exist_ok=True)

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=MAX_STEPS)

if RESUME_FROM and RESUME_STEP > 0:
    opt_path   = os.path.join(RESUME_FROM, 'optimizer.pt')
    sched_path = os.path.join(RESUME_FROM, 'scheduler.json')

    if os.path.exists(opt_path):
        saved_state = torch.load(opt_path, map_location='cpu')
        optimizer.load_state_dict(saved_state)
        for param, state in optimizer.state.items():
            for k, v in state.items():
                if torch.is_tensor(v) and v.device != param.device:
                    state[k] = v.to(param.device)
        print(f'Restored optimizer state from {opt_path}')
    else:
        print(f'WARNING: {opt_path} not found — Adam moments NOT restored.')

    if os.path.exists(sched_path):
        with open(sched_path) as f:
            scheduler.load_state_dict(json.load(f))
        print(f'Restored scheduler state from {sched_path} '
              f'(LR = {scheduler.get_last_lr()[0]:.2e})')
    else:
        print(f'WARNING: {sched_path} not found — scheduler NOT restored.')

    # Guard against resuming with a different MAX_STEPS than the schedule
    # was built for (this bit the old 2.6k run: ckpt@650 schedule, resume@510).
    print(f'Schedule horizon MAX_STEPS={MAX_STEPS}, resume step={RESUME_STEP}, '
          f'last LR={scheduler.get_last_lr()[0]:.2e}')

model.train()
for module in model.modules():
    if 'Visual' in type(module).__name__:
        module.eval()

global_step    = RESUME_STEP
best_loss      = float('inf')
best_ci        = float('inf')
best_ci_ckpt   = None
best_ci_step   = None
no_improve     = 0
session_paused = False
t0             = time.time()
step_times     = []
epoch          = 0
done           = global_step >= MAX_STEPS
ci_history     = []

# Restore early-stop tracker across sessions (fixes old reset-to-inf bug)
_es = load_early_stop_state([
    EARLY_STOP_STATE_PATH,
    os.path.join(RESUME_FROM, 'early_stop_state.json') if RESUME_FROM else None,
    os.path.join(OUTPUT_DIR, 'early_stop_state.json'),
])
if _es:
    _bc = _es.get('best_ci', None)
    best_ci      = float(_bc) if _bc is not None else float('inf')
    best_ci_step = _es.get('best_ci_step')
    no_improve   = int(_es.get('no_improve', 0))
    best_ci_ckpt = _es.get('best_ci_ckpt')
    # If the best ckpt path is from a previous Kaggle working dir, it may be
    # missing; still keep the CI number so we do not re-accept worse models.
    if best_ci_ckpt and not os.path.exists(best_ci_ckpt):
        print(f'NOTE: previous best_ci_ckpt path missing on this machine: '
              f'{best_ci_ckpt} (CI={best_ci}). Will re-copy when a new best lands, '
              f'or fall back to the last improving ckpt this session.')
        best_ci_ckpt = None
    ci_history = list(_es.get('ci_history', []))

print(f'Training: up to {MAX_STEPS} opt steps | starting from step {global_step}')
print(f'Steps remaining this horizon: {MAX_STEPS - global_step}')
print(f'Session stop at: {SESSION_STOP_STEP}')
print(f'Early-stop: patience={EARLY_STOP_PATIENCE}, '
      f'min_steps={MIN_STEPS_BEFORE_EARLY_STOP}, force_full={FORCE_FULL_EPOCH}')
print(f'Current best_ci={best_ci}, no_improve={no_improve}')
print()

if done:
    print('Already at MAX_STEPS — skip to final-save cell.')

while not done:
    epoch += 1
    epoch_loss = 0.0
    n_batches  = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f'Session batches (from step {global_step})')
    for step, batch in enumerate(pbar):
        t_step = time.time()

        pv  = batch.pop('pixel_values').to(dtype).to(LLM_DEVICE)
        thw = batch.pop('image_grid_thw').to(LLM_DEVICE)
        batch = {k: v.to(LLM_DEVICE) for k, v in batch.items()}
        batch['pixel_values']   = pv
        batch['image_grid_thw'] = thw

        outputs = model(**batch)
        loss    = outputs.loss / GRAD_ACCUM
        loss.backward()

        epoch_loss += loss.item() * GRAD_ACCUM
        n_batches  += 1
        step_times.append(time.time() - t_step)

        pbar.set_postfix(
            loss=f'{loss.item()*GRAD_ACCUM:.4f}',
            sps=f'{step_times[-1]:.1f}s',
            step=global_step)

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % LOGGING_STEPS == 0:
                avg  = epoch_loss / n_batches
                lr_n = scheduler.get_last_lr()[0]
                sps  = sum(step_times[-40:]) / len(step_times[-40:])
                eta  = (MAX_STEPS - global_step) * sps * GRAD_ACCUM / 3600
                print(f'Step {global_step:4d}/{MAX_STEPS} | '
                      f'loss={avg:.4f} | lr={lr_n:.2e} | '
                      f'{sps:.1f}s/batch | ETA {eta:.1f}h')

            if global_step % SAVE_STEPS == 0:
                ckpt = os.path.join(OUTPUT_DIR, f'step_{global_step}')
                model.save_pretrained(ckpt)
                processor.save_pretrained(ckpt)
                torch.save(optimizer.state_dict(),
                           os.path.join(ckpt, 'optimizer.pt'))
                with open(os.path.join(ckpt, 'scheduler.json'), 'w') as f:
                    json.dump(scheduler.state_dict(), f)
                # Mirror early-stop state into the ckpt folder for portable resume
                _state = {
                    'best_ci': best_ci if best_ci < float('inf') else None,
                    'best_ci_step': best_ci_step,
                    'best_ci_ckpt': best_ci_ckpt,
                    'no_improve': no_improve,
                    'global_step': global_step,
                    'max_steps': MAX_STEPS,
                    'train_subsample_n': len(records),
                    'seed': SEED,
                    'ci_history': ci_history,
                }
                save_early_stop_state(os.path.join(ckpt, 'early_stop_state.json'), _state)
                save_early_stop_state(EARLY_STOP_STATE_PATH, _state)
                print(f'  ✓ checkpoint → {ckpt} (model + optimizer + scheduler + early_stop)')

                # Scheduled pause for Kaggle 12h — no CI eval (saves ~40 min)
                if global_step >= SESSION_STOP_STEP and global_step < MAX_STEPS:
                    print(f'  ⏸ SESSION_STOP_STEP={SESSION_STOP_STEP} reached. '
                          f'Resume next session with:')
                    print(f'      RESUME_FROM = {ckpt!r}')
                    print(f'      RESUME_STEP = {global_step}')
                    print(f'      SESSION_STOP_STEP = <next pause or MAX_STEPS={MAX_STEPS}>')
                    session_paused = True
                    done = True
                    break

                # ── Dev CI eval + patience-based early stopping ───────────
                metrics = evaluate_ci(model, processor, dev_samples)
                ci = metrics['contrastive_instability']
                ci_history.append({
                    'step': global_step, 'ci': ci,
                    'combined_accuracy': metrics['combined_accuracy'],
                })
                print(f'  → dev CI={ci:.4f} '
                      f'(comb={metrics["combined_accuracy"]:.4f}, '
                      f'cfhr={metrics["cfhr"]:.4f}, '
                      f'q+={metrics["q_plus_accuracy"]:.4f}, '
                      f'q-={metrics["q_minus_accuracy"]:.4f}) '
                      f'| best={best_ci:.4f} | no_improve={no_improve}')

                if ci < best_ci - 1e-12:
                    best_ci      = ci
                    best_ci_ckpt = ckpt
                    best_ci_step = global_step
                    no_improve   = 0
                    print(f'  ✓ new best CI={best_ci:.4f} @ step {best_ci_step}')
                else:
                    no_improve += 1
                    print(f'  · no improvement ({no_improve}/{EARLY_STOP_PATIENCE}) '
                          f'— best remains {best_ci:.4f} @ step {best_ci_step}')

                _state = {
                    'best_ci': best_ci if best_ci < float('inf') else None,
                    'best_ci_step': best_ci_step,
                    'best_ci_ckpt': best_ci_ckpt,
                    'no_improve': no_improve,
                    'global_step': global_step,
                    'max_steps': MAX_STEPS,
                    'train_subsample_n': len(records),
                    'seed': SEED,
                    'ci_history': ci_history,
                }
                save_early_stop_state(os.path.join(ckpt, 'early_stop_state.json'), _state)
                save_early_stop_state(EARLY_STOP_STATE_PATH, _state)

                if (not FORCE_FULL_EPOCH
                        and global_step >= MIN_STEPS_BEFORE_EARLY_STOP
                        and no_improve >= EARLY_STOP_PATIENCE):
                    print(f'  ✗ early stop: {EARLY_STOP_PATIENCE} consecutive '
                          f'non-improving CI evals after step '
                          f'{MIN_STEPS_BEFORE_EARLY_STOP}.')
                    done = True
                    break

            if global_step >= MAX_STEPS:
                done = True
                break

    avg_epoch = epoch_loss / max(n_batches, 1)
    elapsed   = (time.time() - t0) / 3600
    print(f'\nSession pass {epoch} | loss={avg_epoch:.4f} | elapsed={elapsed:.2f}h')
    if avg_epoch < best_loss:
        best_loss = avg_epoch

total_h = (time.time() - t0) / 3600

if session_paused:
    print(f'\nSession paused at step {global_step}. Wall time this session: {total_h:.2f}h')
    print('NOT final — upload checkpoints/ as a Kaggle dataset and resume.')
else:
    print(f'\nTraining finished at step {global_step}/{MAX_STEPS}.')
    print(f'Best train-slice loss: {best_loss:.4f}')
    print(f'Best dev CI: {best_ci:.4f} @ step {best_ci_step} ({best_ci_ckpt})')
    print(f'CI history: {ci_history}')
    print(f'Total this session: {total_h:.2f}h')

    if best_ci_ckpt is not None and os.path.exists(best_ci_ckpt):
        os.makedirs(FINAL_ADAPTER, exist_ok=True)
        for fname in os.listdir(best_ci_ckpt):
            if fname in ('optimizer.pt', 'scheduler.json', 'early_stop_state.json'):
                continue
            shutil.copy2(os.path.join(best_ci_ckpt, fname),
                         os.path.join(FINAL_ADAPTER, fname))
        print(f'Copied best-CI checkpoint ({best_ci_ckpt}) → {FINAL_ADAPTER}')
    else:
        print('No local best-CI checkpoint to copy — final-save cell will '
              'save the in-memory model (or you must re-attach the best ckpt).')


In [ ]:
import glob, zipfile

if 'session_paused' in dir() and session_paused:
    print(f'Session paused at step {global_step} — skipping final adapter zip.')
    print('Upload /kaggle/working/checkpoints as a dataset and resume.')
else:
    if 'best_ci_ckpt' in dir() and best_ci_ckpt is not None and os.path.exists(FINAL_ADAPTER):
        print(f'FINAL_ADAPTER already holds best-CI ckpt ({best_ci_ckpt}).')
    else:
        os.makedirs(FINAL_ADAPTER, exist_ok=True)
        model.save_pretrained(FINAL_ADAPTER)
        processor.save_pretrained(FINAL_ADAPTER)
        print('Saved in-memory model → FINAL_ADAPTER (fallback).')

    # Write a small run card next to the adapter for provenance
    run_card = {
        'run_id': 'RunFT_3k_clean',
        'vlm_model': VLM_MODEL,
        'train_subsample_n': len(records) if 'records' in dir() else TRAIN_SUBSAMPLE_N,
        'sampling': 'nested_prefix',
        'seed': SEED,
        'max_steps': MAX_STEPS,
        'grad_accum': GRAD_ACCUM,
        'learning_rate': LEARNING_RATE,
        'lora_rank': LORA_RANK,
        'max_pixels_train': MAX_PIXELS,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'best_ci': best_ci if 'best_ci' in dir() and best_ci < float('inf') else None,
        'best_ci_step': best_ci_step if 'best_ci_step' in dir() else None,
        'ci_history': ci_history if 'ci_history' in dir() else [],
        'final_step': global_step if 'global_step' in dir() else None,
    }
    with open(os.path.join(FINAL_ADAPTER, 'run_card.json'), 'w') as f:
        json.dump(run_card, f, indent=2)
    print('run_card:', run_card)

    files    = sorted(glob.glob(f'{FINAL_ADAPTER}/*'))
    zip_path = '/kaggle/working/adapter_final.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in files:
            zf.write(f, os.path.basename(f))
            print(f'  {os.path.basename(f):<42} {os.path.getsize(f)/1024/1024:.1f} MB')

    print(f'\nFinal adapter: {zip_path} '
          f'({os.path.getsize(zip_path)/1024/1024:.1f} MB)')
    print('Download from Kaggle output panel, then run inference with CoT5 / 1024px.')


In [ ]:
for i in range(torch.cuda.device_count()):
    a = torch.cuda.max_memory_allocated(i)/1024**3
    r = torch.cuda.max_memory_reserved(i)/1024**3
    t = torch.cuda.get_device_properties(i).total_memory/1024**3
    print(f'GPU {i}: peak_alloc={a:.2f}GB  reserved={r:.2f}GB  total={t:.2f}GB')
